In [ ]:
from nse_platform.context.execution_context import ExecutionContext
from pyspark.sql import functions as F


ec = ExecutionContext.for_local_spark("explore", "2023-05-24")
spark = ec.spark


# NEW UDIFF xplore

In [ ]:
df = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")

In [ ]:

df.printSchema()                       


In [ ]:
df.show(10, truncate=False)              


In [ ]:
print("total rows:", df.count())
df.groupBy("SctySrs").agg(F.count("*").alias("count")).orderBy(F.col("count").desc()).show() 

In [ ]:
df = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")
df= df.filter(F.col("SctySrs").isin("BE","EQ")).na.drop()
df.show()

In [ ]:
COLUMN_MAP_NEW = {
            "TckrSymb": "symbol",
            "SctySrs": "series",
            "TradDt": "trade_date",
            "OpnPric": "open",
            "HghPric": "high",
            "LwPric": "low",
            "ClsPric": "close",
            "LastPric": "last",
            "PrvsClsgPric": "prev_close",
            "TtlTradgVol": "total_traded_qty",
            "TtlTrfVal": "total_traded_value",
            "TtlNbOfTxsExctd": "total_trades",
            "ISIN": "isin",
        }

df = df.withColumnsRenamed(COLUMN_MAP_NEW).select([v for k,v in COLUMN_MAP_NEW.items()])
df.show(10, truncate=False)   

In [ ]:
PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]

df = df.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df = df.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})

In [ ]:
df.select("trade_date").show(5, truncate=False)

In [ ]:
df= df.withColumn("trade_date", F.col("trade_date").cast("date"))

In [ ]:
df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_new_udiff.columns]).show()

In [ ]:
print(df.count())
print(df.na.drop().count())

In [ ]:
df.groupBy("series").count().orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
df.groupBy("symbol", "trade_date").count().filter("count > 1").show()

In [ ]:
df.filter((F.col("symbol") == "RADIOCITY") & (F.col("trade_date") == "2026-01-01")).show(truncate=False)

# FIX

## fix new uidff

In [ ]:

from pyspark.sql import Window

df_new_udiff = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2026/month=01/day=01")
df_new_udiff.filter(F.col("SctySrs").isin("BE","EQ")).na.drop()

df = df_new_udiff.withColumnsRenamed(COLUMN_MAP_NEW).select([v for k,v in COLUMN_MAP_NEW.items()])

PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]
df = df.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df = df.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})
df= df.withColumn("trade_date", F.col("trade_date").cast("date"))


w = Window.partitionBy("symbol", "trade_date").orderBy(F.col("trade_date"))
df = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")
df.show(10, truncate= False)


# explore old udiff

In [ ]:
df_old_udiff = spark.read.parquet("../data/bronze/equity/nse/bhavcopy/year=2023/month=05/day=24")
df_old_udiff.printSchema()

In [ ]:
df_old_udiff.show(10,truncate=False)

In [ ]:
df_old_udiff.filter(F.col("SERIES").isin("BE","EQ")).na.drop().show()

In [ ]:
COLUMN_MAP_OLD = {
            "SYMBOL": "symbol",
            "SERIES": "series",
            "TIMESTAMP": "trade_date",
            "OPEN": "open",
            "HIGH": "high",
            "LOW": "low",
            "CLOSE": "close",
            "LAST": "last",
            "PREVCLOSE": "prev_close",
            "TOTTRDQTY": "total_traded_qty",
            "TOTTRDVAL": "total_traded_value",
            "TOTALTRADES": "total_trades",
            "ISIN": "isin",
        }
df_old_udiff = df_old_udiff.withColumnsRenamed(COLUMN_MAP_OLD).select([v for k,v in COLUMN_MAP_OLD.items()])
df_old_udiff.show(10, truncate=False) 

In [ ]:
print(df_old_udiff.count())
print(df_old_udiff.na.drop().count())

In [ ]:

PRICE_COLS = ["open", "high", "low", "close", "last", "prev_close", "total_traded_value"]
LONG_COLS = ["total_traded_qty", "total_trades"]
df_old_udiff = df_old_udiff.withColumns({c: F.col(c).cast("double") for c in PRICE_COLS})
df_old_udiff = df_old_udiff.withColumns({c: F.col(c).cast("long") for c in LONG_COLS})
df_old_udiff.select("trade_date").show(5, truncate=False)




In [ ]:
df_old_udiff= df_old_udiff.withColumn("trade_date", F.to_date("trade_date", "dd-MMM-yyyy"))

In [ ]:
df_old_udiff.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_old_udiff.columns]).show()

## fix old udiff

In [ ]:
df_old_udiff.filter(F.col("SERIES").isin("BE","EQ")).na.drop()
df_old_udiff = df_old_udiff.withColumnsRenamed(COLUMN_MAP_OLD).select([v for k,v in COLUMN_MAP_OLD.items()])
df_old_udiff= df_old_udiff.withColumn("trade_date", F.to_date("trade_date", "dd-MMM-yyyy"))